In [4]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Directory Setup
base_dir = r"D:\MS_Data_Science_Thesis"
raw_folder = os.path.join(base_dir, r"Data_Extraction\Raw_Data_Folder")
clean_folder = os.path.join(base_dir, r"Data_Cleaning\Semi_Clean_Datasets")
os.makedirs(clean_folder, exist_ok=True)

input_file = os.path.join(raw_folder, "oil_prices_long.csv")

def engineer_wti_features_corrected():
    print("--- Starting Corrected Feature Engineering: WTI Crude Oil ---")
    
    if not os.path.exists(input_file):
        print(f"Error: Could not find {input_file}")
        return

    # 2. Load Data
    df = pd.read_csv(input_file)
    df.columns = df.columns.str.strip()
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').reset_index(drop=True)
    price_col = 'close' if 'close' in df.columns else 'Price'
    
    # 3. Complex Feature Generation (Methodology Aligned)
    print("Calculating Log Returns, MA7, MA50, and 7-Day Volatility...")
    
    # WTI Log Returns
    df['WTI_Log_Return'] = np.log(df[price_col] / df[price_col].shift(1))
    
    # WTI Moving Averages
    df['WTI_MA7'] = df[price_col].rolling(window=7).mean()
    df['WTI_MA50'] = df[price_col].rolling(window=50).mean()
    
    # WTI 7-Day Volatility (Standard Deviation of Log Returns)
    df['WTI_Vol7'] = df['WTI_Log_Return'].rolling(window=7).std()
    
    # 4. Clean Data
    # Drop the first 50 rows that contain NaNs from the MA50 calculation
    df_cleaned = df.dropna(subset=['WTI_MA50', 'WTI_Vol7']).copy()
    
    # 5. Export the intermediate dataset
    output_path = os.path.join(clean_folder, "oil_engineered_III.csv")
    cols_to_keep = ['date', price_col, 'WTI_Log_Return', 'WTI_MA7', 'WTI_MA50', 'WTI_Vol7']
    df_export = df_cleaned[cols_to_keep].rename(columns={price_col: 'WTI_Close'})
    df_export.to_csv(output_path, index=False)
    
    print(f"Success! Corrected WTI dataset saved to: {output_path}")

if __name__ == "__main__":
    engineer_wti_features_corrected()

--- Starting Corrected Feature Engineering: WTI Crude Oil ---


KeyError: 'Date'